Język: [English](lab.en.ipynb) | **Polski**

# Laboratorium 13 — samodzielna implementacja i analiza DDPM (two moons)

Materiał towarzyszący [`README.pl.md`](README.pl.md). Zbiór: two-moons (dwa
półksiężyce) -- inny problem niż wykład (tam: mieszanina Gaussów na okręgu).

In [ ]:
import math
import time

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F  # noqa: F401 - uzywane w TODO ponizej

torch.manual_seed(0)
torch.set_num_threads(4)
device = torch.device("cpu")
print(f"Uzywane urzadzenie: {device}")


def sample_two_moons(n, noise=0.08, seed=None):
    if seed is not None:
        torch.manual_seed(seed)
    n1 = n // 2
    n2 = n - n1
    theta1 = torch.rand(n1) * math.pi
    moon1 = torch.stack([torch.cos(theta1), torch.sin(theta1)], dim=1)
    theta2 = torch.rand(n2) * math.pi
    moon2 = torch.stack([1 - torch.cos(theta2), 1 - torch.sin(theta2) - 0.5], dim=1)
    X = torch.cat([moon1, moon2], dim=0)
    return (X + noise * torch.randn(X.shape[0], 2)) * 3.0


X_train = sample_two_moons(2000, seed=0)
print(f"X_train: {X_train.shape}, mean={X_train.mean(0)}, std={X_train.std(0)}")

## 1. Wizualizacja danych — Twoje zadanie

Narysuj wykres rozrzutu `X_train`. W komentarzu/markdown: czym ten kształt
różni się od mieszaniny Gaussów z wykładu, i dlaczego jeden rozkład Gaussa
nie mógłby go dobrze opisać?

In [ ]:
# TODO: plt.scatter(...) dla X_train, plt.axis("equal"), plt.show()
raise NotImplementedError("TODO: wizualizacja danych")

## 2. Harmonogram β i obliczanie ᾱ_t — Twoje zadanie

In [ ]:
T = 500
BETA_START, BETA_END = 1e-4, 0.02

# TODO:
# betas = torch.linspace(BETA_START, BETA_END, T)
# alphas = 1 - betas
# alphas_bar = torch.cumprod(alphas, dim=0)
# wypisz abar na t=1, T//2, T -- czy abar_T jest wystarczajaco blisko 0?
raise NotImplementedError("TODO: harmonogram i alphas_bar")

## 3. Zamknięta postać forward diffusion — Twoje zadanie

Zaimplementuj `q_sample` i zweryfikuj ją statystycznie (średnia/wariancja na
wielu próbach `eps`) względem symulacji krok-po-kroku -- **nie** oczekuj
identyczności pojedynczych próbek.

In [ ]:
# TODO:
# def q_sample(x0, t, eps, abar):
#     ...
#
# WERYFIKACJA (wymagana): dla ustalonego x0 i t, porownaj mean/var miedzy
# q_sample (N probek eps) i symulacja krok-po-kroku (N niezaleznych lancuchow)
raise NotImplementedError("TODO: q_sample + weryfikacja statystyczna")

## 4. Sieć przewidująca szum i warunkowanie czasem — Twoje zadanie

`sinusoidal_embedding` jest gotowa. `NoisePredictor.forward` musi ją
wykorzystać -- skonkatenować z `x` przed warstwami liniowymi.

In [ ]:
def sinusoidal_embedding(t, dim=16):
    half = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half).float() / half)
    args = t.float().unsqueeze(1) * freqs.unsqueeze(0)
    return torch.cat([torch.sin(args), torch.cos(args)], dim=1)

In [ ]:
# TODO:
# class NoisePredictor(nn.Module):
#     def __init__(self, time_emb_dim=16, hidden=128):
#         ...  # np. Linear(2+time_emb_dim,h)->ReLU->Linear(h,h)->ReLU->Linear(h,2)
#     def forward(self, x, t):
#         ...  # MUSI uzyc sinusoidal_embedding(t, ...) i polaczyc z x
#
# Komentarz: dlaczego x_t samo nie wystarcza modelowi bez znajomosci t?
raise NotImplementedError("TODO: siec + warunkowanie czasem")

## 5. Funkcja celu i pętla treningowa — Twoje zadanie

Szkielet pętli (optymalizator, iteracja po epokach/batchach) jest gotowy w
`train_ddpm` -- Twoje zadanie jest **wewnątrz** pętli (krok treningowy).

In [ ]:
def train_ddpm(model, X, T_steps, abar, epochs=300, batch_size=128, lr=1e-3, seed=0):
    torch.manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=lr)  # noqa: F841 - uzywane w TODO ponizej
    n = X.shape[0]
    losses = []
    t0 = time.time()
    for epoch in range(epochs):
        perm = torch.randperm(n)
        epoch_loss = 0.0
        for i in range(0, n, batch_size):
            idx = perm[i : i + batch_size]
            x0_batch = X[idx]
            b = x0_batch.shape[0]  # noqa: F841 - uzywane w TODO ponizej
            # TODO:
            # 1. t = torch.randint(0, T_steps, (b,))
            # 2. eps = torch.randn(b, 2)
            # 3. x_t = q_sample(x0_batch, t, eps, abar)
            # 4. eps_pred = model(x_t, t)
            # 5. loss = F.mse_loss(eps_pred, eps)
            # 6. opt.zero_grad(); loss.backward(); opt.step()
            # 7. epoch_loss += loss.item() * b
            raise NotImplementedError("TODO: krok treningowy DDPM")
        losses.append(epoch_loss / n)
    return time.time() - t0, losses

In [ ]:
model = NoisePredictor()  # noqa: F821 - zdefiniowane w sekcji 4
train_time, losses = train_ddpm(model, X_train, T, alphas_bar)  # noqa: F821
print(f"train_time: {train_time:.2f}s")

plt.figure(figsize=(5, 3.5))
plt.plot(losses)
plt.xlabel("epoka")
plt.ylabel("L_simple")
plt.title("Strata treningowa (monitorowanie)")
plt.tight_layout()
plt.show()

## 6. Odwrotne próbkowanie — Twoje zadanie

In [ ]:
def ddpm_reverse_step(model, x, t_idx, alphas, betas, alphas_bar):
    t_batch = torch.full((x.shape[0],), t_idx, dtype=torch.long)
    eps_pred = model(x, t_batch)  # noqa: F841 - uzywane w TODO ponizej
    alpha_t, beta_t, abar_t = alphas[t_idx], betas[t_idx], alphas_bar[t_idx]  # noqa: F841
    # TODO:
    # mu = (1 / torch.sqrt(alpha_t)) * (x - (beta_t / torch.sqrt(1 - abar_t)) * eps_pred)
    # if t_idx > 0: return mu + torch.sqrt(beta_t) * torch.randn_like(x)
    # else: return mu  # OSTATNI krok: bez dodanego szumu
    raise NotImplementedError("TODO: krok odwrotny DDPM")


@torch.no_grad()
def ddpm_sample(model, n_samples, T_steps, alphas, betas, alphas_bar, seed=0):
    torch.manual_seed(seed)
    x = torch.randn(n_samples, 2)
    n_with_noise, n_without_noise = 0, 0
    for t_idx in reversed(range(T_steps)):
        x = ddpm_reverse_step(model, x, t_idx, alphas, betas, alphas_bar)
        if t_idx > 0:
            n_with_noise += 1
        else:
            n_without_noise += 1
    print(
        f"kroki z dodanym szumem: {n_with_noise}, bez szumu: {n_without_noise} "
        f"(oczekiwane: {T_steps - 1} i 1)"
    )
    return x

In [ ]:
model.eval()
t0 = time.time()
samples = ddpm_sample(model, 1000, T, alphas, betas, alphas_bar, seed=42)  # noqa: F821
print(f"generation time: {time.time() - t0:.2f}s")

## 7. Generowanie nowych próbek i porównanie rozkładów — Twoje zadanie

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].scatter(X_train[:, 0], X_train[:, 1], s=4, alpha=0.4)
axes[0].set_title("Dane treningowe")
axes[0].axis("equal")
axes[1].scatter(samples[:, 0], samples[:, 1], s=4, alpha=0.4, color="tab:orange")
axes[1].set_title("Wygenerowane probki (od czystego szumu, NIE rekonstrukcje)")
axes[1].axis("equal")
plt.tight_layout()
plt.show()

In [ ]:
def rbf_mmd2(X, Y, sigma=None):
    XX, YY, XY = torch.cdist(X, X) ** 2, torch.cdist(Y, Y) ** 2, torch.cdist(X, Y) ** 2
    if sigma is None:
        combined = torch.cat([XX.flatten(), YY.flatten(), XY.flatten()])
        sigma = combined[combined > 0].median().sqrt().item()
    gamma = 1.0 / (2 * sigma**2)
    Kxx, Kyy, Kxy = torch.exp(-gamma * XX), torch.exp(-gamma * YY), torch.exp(-gamma * XY)
    nx, ny = X.shape[0], Y.shape[0]
    sum_xx = (Kxx.sum() - Kxx.diag().sum()) / (nx * (nx - 1))
    sum_yy = (Kyy.sum() - Kyy.diag().sum()) / (ny * (ny - 1))
    sum_xy = Kxy.sum() / (nx * ny)
    return (sum_xx + sum_yy - 2 * sum_xy).item(), sigma


def mean_nn_distance(samples, reference):
    return torch.cdist(samples, reference).min(dim=1).values.mean().item()

In [ ]:
# WERYFIKACJA MMD na znanym przypadku -- PRZED uzyciem na wygenerowanych danych
torch.manual_seed(1)
A = torch.randn(300, 2)
B_same = torch.randn(300, 2)
B_shifted = torch.randn(300, 2) + torch.tensor([5.0, 5.0])
mmd_same, sigma_ref = rbf_mmd2(A, B_same)
mmd_diff, _ = rbf_mmd2(A, B_shifted, sigma=sigma_ref)
print(f"MMD^2 kontrolne: ta sama dystrybucja={mmd_same:.5f}, przesunieta={mmd_diff:.5f}")
assert mmd_same < 0.01
assert mmd_diff > mmd_same * 10

# TODO:
# X_test = sample_two_moons(1000, seed=123)  # NIEZALEZNY zbior testowy
# mmd_gen, sigma_data = rbf_mmd2(X_test, samples)
# nn_gen = mean_nn_distance(samples, X_test)
# wypisz obie metryki
raise NotImplementedError("TODO: ocena generowania na niezaleznym zbiorze testowym")

## 8. Sprawdzenie jednej zmiany konfiguracji — Twoje zadanie

Wybierz **jedną** zmianę (liczba kroków próbkowania z poprawnym respacing,
harmonogram, albo liczba epok treningu) i zmierz wpływ tą samą metryką.

In [ ]:
# TODO: wybierz JEDNA zmiane, zmierz te sama metryke (mmd_gen albo nn_gen z sekcji 7)
raise NotImplementedError("TODO: jedna zmiana konfiguracji")

## 9. Baseline — Twoje zadanie

Pojedynczy dopasowany rozkład Gaussa. Nie zakładaj z góry, który model
wygra na każdej metryce.

In [ ]:
# TODO:
# mu_fit = X_train.mean(0)
# cov_fit = torch.cov(X_train.T)
# baseline_dist = torch.distributions.MultivariateNormal(mu_fit, cov_fit)
# baseline_samples = baseline_dist.sample((1000,))
# policz mmd/nn-distance dla baseline_samples wzgledem X_test, porownaj z sekcja 7
raise NotImplementedError("TODO: baseline Gaussa")

## 10. Zadanie diagnostyczne

Poniższy kod (gotowy, uruchom go) implementuje `NoisePredictorColleague` --
przyjmuje `t`, ale **nigdy go nie używa** w `forward`.

In [ ]:
class NoisePredictorColleague(nn.Module):
    def __init__(self, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 2),
        )

    def forward(self, x, t):
        return self.net(x)


model_colleague = NoisePredictorColleague()
_, losses_colleague = train_ddpm(model_colleague, X_train, T, alphas_bar)  # noqa: F821
samples_colleague = ddpm_sample(  # noqa: F821
    model_colleague,
    1000,
    T,
    alphas,  # noqa: F821
    betas,  # noqa: F821
    alphas_bar,  # noqa: F821
    seed=42,
)
print(f"final loss (colleague, no time conditioning): {losses_colleague[-1]:.4f}")
print(f"final loss (Twoj model, sekcja 5): {losses[-1]:.4f}")  # noqa: F821

In [ ]:
# TODO:
# 1. policz mmd/nn-distance dla samples_colleague wzgledem X_test (sekcja 7)
# 2. porownaj z Twoimi wynikami z sekcji 7
# 3. w komentarzu: wyjasnij DLACZEGO model bez t radzi sobie gorzej, mimo ze
#    wciaz cos generuje (nie crashuje)
raise NotImplementedError("TODO: diagnoza braku warunkowania czasem")

## 11. Wnioski — Twoje zadanie

**1. Czy ᾱ_T dla Twojego harmonogramu było wystarczająco bliskie 0? Co by się stało, gdyby nie było?**

_(Twoja odpowiedź tutaj.)_

**2. Jak dobrze (konkretne metryki) Twój model odtwarza rozkład two-moons na niezależnym zbiorze testowym?**

_(Twoja odpowiedź tutaj.)_

**3. Jak wypada w porównaniu z baseline'em? Czy obie metryki dały tę samą odpowiedź?**

_(Twoja odpowiedź tutaj.)_

**4. Jaki efekt miała zmiana konfiguracji z sekcji 8?**

_(Twoja odpowiedź tutaj.)_

**5. Jaki błąd wprowadził brak warunkowania czasem, i jaka jest ogólna zasada?**

_(Twoja odpowiedź tutaj.)_

**6. Są obszary przestrzeni danych, których model nie generuje dobrze? Jak to sprawdziłeś/aś?**

_(Twoja odpowiedź tutaj.)_